# 📬 Nyaya-Jyoti: AI-Powered Friendly Loan Legal Notice Generator

Welcome to **Nyaya-Jyoti**, your intelligent legal assistant that helps you generate **personalized loan recovery notices** in just a few steps — with no legal drafting expertise needed.

This tool uses **GPT-Neo AI** and **Semantic Matching** to generate legally valid clauses, including **custom penalty or interest demands**. Your final notice is auto-generated based on your answers and ready to send.

---

### 📝 How it works:

1. **Upload your clause CSV** and the **Legal Notice Template (.docx)**
2. **Answer simple questions** about the lender, borrower, loan amount, and dates
3. **Add any special clause** (e.g., interest claim or compensation request) — AI will draft it for you
4. **Watch the notice update live** with your answers and AI-drafted text
5. **Download your final notice** as a `.docx` file, ready to print or email

---

### ⚠️ Instructions:

- Be ready to upload:
  - ✅ Clause prompt CSV file for AI to generate legal clauses
  - ✅ Word (.docx) notice template with placeholders like `[Lender_Name]`, `[Loan_Amount]`, etc.
- Provide all dates in `DD/MM/YYYY` format and write names/address details carefully
- Special clause (interest, penalty, legal demand) is handled after standard fields
- The final `.docx` notice will auto-download after completion

---

> 🛡️ This tool is part of the *Nyaya-Jyoti* legal innovation project from Bennett University. This prototype is for demonstration and should be reviewed before actual use.



In [ ]:
# @title
# 🛠️ Install required packages
!pip install -q transformers torch pandas sentence-transformers python-docx

# 📦 Imports
import torch
import pandas as pd
import re
import docx
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
from google.colab import files
from IPython.display import display, Markdown

# --- Load Clause Prompt Registry ---
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
prompt_df = pd.read_csv(csv_filename)
prompt_df["parameters"] = prompt_df["parameters"].fillna("").astype(str)
prompt_df["parameters"] = prompt_df["parameters"].apply(
    lambda x: ", ".join(sorted(set(p.strip() for p in x.split(",") if p.strip().lower() != "nan")))
)

# --- Load GPT-Neo & Embedding Model ---
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
instruction_embeddings = embedder.encode(prompt_df['instruction'].tolist(), convert_to_tensor=True)

# --- Clause Generation Utilities ---
def build_prompt(example_1, example_2, instruction):
    return (
        "You are a legal assistant specialized in drafting formal legal clauses.\n"
        f"Example 1:\nClause: {example_1}\nEndClause\n\n"
        f"Example 2:\nClause: {example_2}\nEndClause\n\n"
        f"Now, generate ONLY the legal clause for {instruction}, using formal legal language.\n"
        "Output only the text between the markers 'Clause:' and 'EndClause'.\n\nClause: "
    )

def generate_clause(prompt_text):
    input_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    output_ids = model.generate(
        input_ids,
        max_length=300,
        temperature=0.35,
        top_k=50,
        top_p=0.85,
        repetition_penalty=1.2,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id
    )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "EndClause" in generated_text:
        return generated_text.split("Clause:")[1].split("EndClause")[0].strip()
    return generated_text.split("Clause:")[1].strip()

def fill_parameters_dynamic(clause_text, param_string):
    placeholders = set(re.findall(r"{(.*?)}", clause_text))
    defined_params = [p.strip() for p in str(param_string).split(',') if p.strip()]
    combined_params = sorted(placeholders.union(set(defined_params)))
    param_values = {}
    for param in combined_params:
        value = input(f"🧾 Please provide value for '{param}': ").strip()
        param_values[param] = value
    for param, value in param_values.items():
        clause_text = clause_text.replace(f"{{{param}}}", value)
    return clause_text, param_values

def find_best_match_semantic(user_input):
    user_embedding = embedder.encode(user_input, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(user_embedding, instruction_embeddings)[0]
    best_idx = torch.argmax(cosine_scores).item()
    best_score = cosine_scores[best_idx].item()
    if best_score > 0.4:
        return prompt_df.iloc[best_idx]
    return None

# --- Upload Template ---
print("📂 Upload your Friendly Loan Legal Notice DOCX template:")
uploaded_template = files.upload()
template_filename = list(uploaded_template.keys())[0]
doc = docx.Document(template_filename)

# --- Placeholder Explanations (Updated) ---
explanations = {
    "Lender_Name": "Enter the full name of the lender sending the notice",
    "Lender_Address": "Enter the full address of the lender (e.g., house no., street, locality)",
    "City": "Enter the city",
    "State": "Enter the state",
    "PIN_Code": "Enter the postal code",
    "Email_Address": "Enter the lender's email address",
    "Phone_Number": "Enter the lender's phone number",
    "Date": "Enter the date of the notice (e.g., 20/04/2025)",
    "Borrower_Name": "Enter the full name of the borrower",
    "Borrower_Address": "Enter the full address of the borrower (e.g., house no., street, locality)",
    "Loan_Amount": "Enter the amount of the loan in numbers (e.g., 100000)",
    "Loan_Amount_In_Words": "Enter the amount in words (e.g., one lakh)",
    "Loan_Date": "Enter the date when the loan was given (e.g., 20/04/2024)",
    "Due_Date": "Enter the due date of repayment (e.g., 20/04/2025)",
    "Interest_Rate": "Enter the interest rate in % (e.g., 10) or 0 if none",
    "Special_Clauses": "Special terms (e.g., demand for interest or penalty)"
}

# --- Collect & Replace All Placeholders ---
print("\n📋 Please answer the following questions to populate the notice:")
user_inputs = {}
for placeholder, explanation in explanations.items():
    if placeholder == "Special_Clauses":
        continue
    value = input(f"🖋 {explanation}: ").strip()
    user_inputs[placeholder] = value

# --- Replace All Placeholders in Paragraphs (multi-token line safe) ---
for para in doc.paragraphs:
    original_text = para.text
    modified_text = original_text
    for placeholder, value in user_inputs.items():
        modified_text = modified_text.replace(f"[{placeholder}]", value)
    if modified_text != original_text:
        para.text = modified_text
        display(Markdown(f"**📄 Updated Line:**\n\n`Before:` {original_text}\n\n`After:` {modified_text}"))

# --- Special Clause Prompt Comes AFTER Placeholder Filling ---
special_clause_text = ""
print("\n📄 All standard fields are now filled.")
special_query = input("\n💬 Do you have any special clause to include (e.g., 'add demand for interest or compensation')?\n📨 Special Request (leave blank if none): ").strip()
if special_query:
    match_row = find_best_match_semantic(special_query)
    if match_row is not None:
        prompt = build_prompt(match_row['example_1'], match_row['example_2'], match_row['instruction'])
        raw_clause = generate_clause(prompt)
        special_clause_text, _ = fill_parameters_dynamic(raw_clause, match_row.get('parameters', ''))
    else:
        # fallback to directly generating from input
        prompt = (
            "You are a legal assistant. Based on the request below, draft a formal clause:\n\n"
            f"Request: {special_query}\n\nClause:"
        )
        raw_clause = generate_clause(prompt)
        special_clause_text, _ = fill_parameters_dynamic(raw_clause, "")

# --- Insert Special Clause in Template ---
if special_clause_text:
    for para in doc.paragraphs:
        if "[Special_Clauses]" in para.text:
            para.text = para.text.replace("[Special_Clauses]", special_clause_text)
            display(Markdown(f"**📄 Inserted Special Clause:** {para.text}"))
            break

# --- Save Final Notice ---
output_file = "completed_friendly_loan_notice.docx"
doc.save(output_file)
files.download(output_file)
print(f"\n✅ Notice saved as: {output_file}")


Saving Friendly_Loan_Notice_Clauses.csv to Friendly_Loan_Notice_Clauses (2).csv
📂 Upload your Friendly Loan Legal Notice DOCX template:


Saving friendly_loan_notice_template.docx to friendly_loan_notice_template (2).docx

📋 Please answer the following questions to populate the notice:
🖋 Enter the full name of the lender sending the notice: Amit
🖋 Enter the full address of the lender (e.g., house no., street, locality): xyz road
🖋 Enter the city: Noida
🖋 Enter the state: Uttarpradesh
🖋 Enter the postal code: 201009
🖋 Enter the lender's email address: XYZ@gmail.com
🖋 Enter the lender's phone number: 665615161651
🖋 Enter the date of the notice (e.g., 20/04/2025): 20/04/2025
🖋 Enter the full name of the borrower: sumit
🖋 Enter the full address of the borrower (e.g., house no., street, locality): abc road
🖋 Enter the amount of the loan in numbers (e.g., 100000): 10000
🖋 Enter the amount in words (e.g., one lakh): one lakh
🖋 Enter the date when the loan was given (e.g., 20/04/2024): 20/04/2025
🖋 Enter the due date of repayment (e.g., 20/04/2025): 
🖋 Enter the interest rate in % (e.g., 10) or 0 if none: 


**📄 Updated Line:**

`Before:` [Lender_Name]
[Lender_Address]
[City, State, PIN Code]
[Email Address]
[Phone Number]
[Date]

`After:` Amit
xyz road
[City, State, PIN Code]
[Email Address]
[Phone Number]
20/04/2025

**📄 Updated Line:**

`Before:` To,
[Borrower_Name]
[Borrower_Address]
[City, State, PIN Code]

`After:` To,
sumit
abc road
[City, State, PIN Code]

**📄 Updated Line:**

`Before:` Dear [Borrower_Name],

`After:` Dear sumit,

...

**📄 Updated Line:**

`Before:` Yours sincerely,
[Lender_Signature]
[Lender_Name]

`After:` Yours sincerely,
[Lender_Signature]
Amit


📄 All standard fields are now filled.

💬 Do you have any special clause to include (e.g., 'add demand for interest or compensation')?
📨 Special Request (leave blank if none): 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Notice saved as: completed_friendly_loan_notice.docx
